# 1 · Masked Sentinel-2 composite for Tongatapu

The Python counterpart of GEE **script 1**. It builds a cloud-free composite of six months of dry-season Sentinel-2 imagery, then masks out land and optically deep water, leaving the shallow areas to classify.

| Step | GEE script 1 | Here |
|---|---|---|
| Find scenes | `COPERNICUS/S2_SR_HARMONIZED`, `CLOUDY_PIXEL_PERCENTAGE ≤ 30` | STAC search (Element84 Earth Search), `eo:cloud_cover ≤ 30` |
| Cloud and shadow mask | s2cloudless probability + projected shadows, 50 m buffer | Sen2Cor scene classification (`SCL`) cloud and shadow classes, 50 m buffer |
| One image per date | `mosaicBySolarDay` | `groupby="solar_day"` when loading |
| Composite | 25th percentile | 25th percentile |
| Land mask | MNDWI, Otsu threshold near land/water edges | same |
| Deep water mask | hue/saturation index, Otsu near edges, 250 m dilation | same index, with a fixed threshold you tune for each scene |

**Output:** `data/tongatapu_composite.nc`

*Deep water:* Lee, C.B., Traganos, D. & Reinartz, P. (2022) A simple cloud-native spectral transformation method to disentangle optically shallow and deep waters in Sentinel-2 images. *Remote Sensing* 14: 590.

The composite reads six months of imagery, a few GB, so the first run takes a while. Later notebooks only read the saved composite.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray  # noqa: F401  (adds .rio to xarray objects)
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import odc.stac
from pystac_client import Client
from dask.diagnostics import ProgressBar
from scipy import ndimage
from skimage.feature import canny
from skimage.filters import threshold_otsu
from skimage.color import rgb2hsv

AOI = (-175.36, -21.20, -174.99, -20.99)   # lon/lat: west, south, east, north
START, END = "2023-05-01", "2023-10-31"   # dry season
MAX_CLOUD = 30            # % cloud per scene (GEE: CLOUD_FILTER)
CRS = "EPSG:32701"        # UTM zone 1S
RES = 10                  # m

STAC_URL = "https://earth-search.aws.element84.com/v1"
COLLECTION = "sentinel-2-c1-l2a"
BANDS = {"B1": "coastal", "B2": "blue", "B3": "green", "B4": "red", "B5": "rededge1", "B8A": "nir08", "B11": "swir16"}

OUT = Path("data") / "tongatapu_composite.nc"
OUT.parent.mkdir(exist_ok=True)

## 1. Find the scenes

The STAC search returns one item per Sentinel-2 tile and date, with metadata such as the scene's cloud cover.

In [ ]:
items = Client.open(STAC_URL).search(collections=[COLLECTION], bbox=AOI, datetime=f"{START}/{END}",
                                     query={"eo:cloud_cover": {"lte": MAX_CLOUD}}).item_collection()
dates = pd.Series([pd.Timestamp(it.datetime).date() for it in items])
print(f"{len(items)} items on {dates.nunique()} dates with cloud ≤ {MAX_CLOUD}%")
dates.value_counts().sort_index().groupby(lambda d: d.strftime("%Y-%m")).size().plot.bar(
    figsize=(9, 2.5), title="Dates per month", color="0.5");

## 2. Load the data lazily

`odc.stac.load` puts every scene on the same 10 m UTM grid and merges tiles from the same overpass (`groupby="solar_day"`). With `chunks`, nothing is downloaded yet: we get a **lazy** array that is only read, piece by piece, when the composite is computed.

In [ ]:
ds = odc.stac.load(items, bands=list(BANDS.values()) + ["scl"], bbox=AOI, crs=CRS, resolution=RES,
                   groupby="solar_day", chunks={"x": 1024, "y": 1024})
ds = ds.rename({v: k for k, v in BANDS.items()} | {"scl": "SCL"})
print(f"{ds.sizes['time']} dates, {ds.sizes['y']} × {ds.sizes['x']} pixels, "
      f"{ds.nbytes / 1e9:.1f} GB if loaded in full")

## 3. Mask clouds and shadows

Sen2Cor classifies every pixel (`SCL`). We mask no data, saturated pixels, cloud shadow, medium and high cloud, and cirrus, then grow the mask by 50 m to catch cloud edges. A 50 m buffer is a moving maximum over an 11 × 11 pixel window.

Digital numbers become reflectance as `DN / 10000 − 0.1`. The −0.1 offset applies to all scenes processed since January 2022, which includes this whole period.

In [ ]:
SCL_MASK = [0, 1, 3, 8, 9, 10]        # no data, saturated, cloud shadow, cloud medium, cloud high, cirrus
BUFFER_M = 50
window = 2 * round(BUFFER_M / RES) + 1

cloud = ds["SCL"].isin(SCL_MASK).astype("uint8").rolling(x=window, y=window, center=True, min_periods=1).max() == 1
bands = ds[list(BANDS)]
refl = (bands.where(bands > 0) / 10000 - 0.1).where(~cloud)

In [ ]:
def rgb(img, gain):
    return np.clip(np.nan_to_num(np.dstack([img["B4"], img["B3"], img["B2"]]) * gain), 0, 1)

cloud_by_date = pd.Series({pd.Timestamp(it.datetime).date(): it.properties["eo:cloud_cover"] for it in items})
median_day = cloud_by_date.sort_values().index[len(cloud_by_date) // 2]          # a scene with typical cloud
t = int(np.flatnonzero(pd.to_datetime(ds.time.values).date == median_day)[0])
before = (bands.isel(time=t) / 10000 - 0.1).coarsen(x=8, y=8, boundary="trim").mean().compute()
after = refl.isel(time=t).coarsen(x=8, y=8, boundary="trim").mean().compute()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, img, title in [(axes[0], before, "as delivered"), (axes[1], after, "cloud and shadow masked")]:
    ax.imshow(rgb(img, 4)); ax.set_title(f"{pd.Timestamp(ds.time.values[t]):%Y-%m-%d}: {title}"); ax.axis("off")

## 4. 25th percentile composite

For each pixel and band, the 25th percentile of all cloud-free observations. A low percentile rejects leftover cloud, haze and glint, which make pixels brighter, without taking the very darkest value, which is often an unmasked shadow.

Over dark water, atmospheric correction can leave reflectance slightly **below zero**. That has no physical meaning, and it breaks ratio indices: when $G + \text{dark}$ is near zero, the index explodes. So the composite is clipped at 0.

Computing the percentile needs all dates for a pixel at once, so the data are rechunked along time first. This step downloads the imagery.

In [ ]:
with ProgressBar():
    comp = refl.chunk(time=-1).quantile(0.25, dim="time", skipna=True).drop_vars("quantile").compute()
comp = comp.astype("float32").clip(min=0)    # atmospheric correction can leave dark water slightly below 0

fig, ax = plt.subplots(figsize=(12, 9))
ax.imshow(rgb(comp.coarsen(x=4, y=4, boundary="trim").mean(), 4)); ax.set_title("25th percentile composite"); ax.axis("off");

## 5. Land mask

Water index: green against the darker of SWIR (B11) and NIR (B8A). That is MNDWI where SWIR is darker, and NDWI where NIR is.

A single global threshold rarely fits every scene, so the threshold is found with **Otsu's method**, using only pixels **near land/water edges**, where both classes are well represented:
1. Split the image at a rough initial threshold
2. Find the edges of that split (Canny), and keep only long edges (≥ 80 pixels), not speckle
3. Take the pixels within ~90 m of those edges and compute the Otsu threshold of their index values

(GEE: `fastDistanceTransform().lt(80)` is a *squared* distance in pixels, about 9 pixels or 90 m.)

In [ ]:
def edge_otsu(index, initial_threshold, min_edge_px, buffer_px):
    """Otsu threshold of the index, using only pixels near long edges of an initial split."""
    values = index.values
    ok = np.isfinite(values)                                   # NaN areas (masked, no data) are ignored
    split = np.where(ok, values, initial_threshold) < initial_threshold
    edges = canny(split.astype(float), sigma=1, mask=ok)
    labels, n = ndimage.label(edges, structure=np.ones((3, 3)))
    length = ndimage.sum(edges, labels, index=np.arange(1, n + 1))
    long_edges = np.isin(labels, np.flatnonzero(length >= min_edge_px) + 1)
    near = ndimage.distance_transform_edt(~long_edges) <= buffer_px
    return threshold_otsu(values[near & ok]), near

dark = np.minimum(comp["B11"], comp["B8A"])
mndwi = (comp["B3"] - dark) / (comp["B3"] + dark)
land_threshold, land_edges = edge_otsu(mndwi, 0, 80, round(90 / RES))
water = (mndwi >= land_threshold) & comp["B2"].notnull()
print(f"land/water threshold: {land_threshold:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 5), gridspec_kw={"width_ratios": [0.7, 1]})
v = mndwi.values
axes[0].hist(v[np.isfinite(v)], bins=200, color="0.8", density=True, label="all pixels")
axes[0].hist(v[land_edges & np.isfinite(v)], bins=200, histtype="step", color="C0", density=True, label="near edges")
axes[0].axvline(land_threshold, color="r", label=f"Otsu threshold {land_threshold:.2f}")
axes[0].set(xlabel="water index", yticks=[]); axes[0].legend()
axes[1].imshow(rgb(comp.where(water), 4)); axes[1].set_title("land masked"); axes[1].axis("off")
plt.tight_layout()

## 6. Optically deep water mask

Where the water is too deep for light to return from the bottom, there is nothing to classify. Brightness alone can't find these areas: clear oceanic water is *bright* in blue, because water scatters blue light, while dense seagrass in shallow water is dark in every band. What differs is the **shape** of the spectrum. Deep water is dark in green, and over any bottom the green signal rises relative to blue.

As in GEE script 1 (after Lee et al. 2022), we treat coastal blue (B1), blue (B2) and green (B3) as an RGB colour and convert it to hue $H$ and saturation $S$. Both are independent of brightness. The **deep water index** is

$$DWI = \frac{S - H}{S + H}$$

Deep water is strongly saturated and blue-shifted, so its DWI is high; shallow water has a lower DWI. Lee et al. found that an automatic (Otsu) threshold did not work for this index, and set it from labelled data instead. Here, **adjust `DEEP_THRESHOLD`** using the histogram and map below, and check it against the drop-camera data in notebook 2. Lowering the threshold masks more water.

Anything deep that the mask misses is caught in notebook 2 by the `Deep Water` class.

In [ ]:
DEEP_THRESHOLD = 0.2        # deep where DWI is above this; tune for each composite

hsv = rgb2hsv(np.clip(np.nan_to_num(np.dstack([comp["B1"], comp["B2"], comp["B3"]])), 0, 1))
dwi = comp["B3"].copy(data=(hsv[..., 1] - hsv[..., 0]) / (hsv[..., 1] + hsv[..., 0])).where(water)
deep = (dwi > DEEP_THRESHOLD) & water
shallow = water & ~deep
print(f"threshold {DEEP_THRESHOLD}: {float(deep.sum() / water.sum()):.0%} of water masked as optically deep")

fig, axes = plt.subplots(1, 2, figsize=(17, 5), gridspec_kw={"width_ratios": [0.7, 1]})
v = dwi.values[np.isfinite(dwi.values)]
axes[0].hist(v, bins=200, color="0.6")
axes[0].axvline(DEEP_THRESHOLD, color="r", label=f"DEEP_THRESHOLD = {DEEP_THRESHOLD}")
axes[0].set(xlabel="deep water index, water pixels", yticks=[]); axes[0].legend()
axes[1].imshow(rgb(comp.where(water), 4))
axes[1].imshow(np.where(deep, 1.0, np.nan), cmap=ListedColormap(["#08306b"]), alpha=0.7)
axes[1].set_title("optically deep (dark blue)"); axes[1].axis("off")
plt.tight_layout()

## 7. Save

The composite is saved with its masks (`water`, and `shallow` = water that is not optically deep) and both thresholds. Reflectance is stored as integers × 10 000 to keep the file small.

In [ ]:
out = comp.assign(water=water.astype("uint8"), shallow=shallow.astype("uint8")).rio.write_crs(CRS)
out.attrs.update(land_threshold=float(land_threshold), deep_threshold=float(DEEP_THRESHOLD),
                 dates=f"{START} to {END}", n_dates=int(ds.sizes["time"]))
encoding = {b: {"dtype": "int16", "scale_factor": 1e-4, "_FillValue": -32768, "zlib": True} for b in BANDS}
OUT.unlink(missing_ok=True)
out.to_netcdf(OUT, engine="h5netcdf", encoding=encoding)
print(f"saved {OUT}")